**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

Our stock picking strategy follows these stages:

#### Stage 1 - Market Stats

**What it does:** We download data of the S&P 500 over the same period as our stock data. The closing prices of index represent the market and are used to calculate beta

The period we chose was: 2023-10-02 to 2026-09-30

We chose the S&P500 since it is a very broad/diversified, market-cap-weighted index. It also represents the Canada and US markets overall pretty well (where all the stocks are coming from): while all S&P500 stocks are US-based, the US market makes up the majority of the equity value in these two countries, so it is a reasonable choice. 

#### Stage 2 - Statistics for every stock

**What it does:** For every eligible stock we calculate 4 statistics:
- Standard deviation of daily returns
- Beta of the stock's daily returns vs. market returns
- Crash count: how many times the stock fell by 20% in price within 5-trading-day trading periods, over our data period
- Market cap: measured in CAD on the day trading starts

**Why these statistics:**

- Std: This is what the client uses to define risk, so we start with this calculation. However, past std does not always reflect future stock volatility, so we also need to consider other risk indicators.
- Beta: While we can diversify our stocks to remove industry or business risk, this still would not remove market risk since all the stocks chosen are subject to shocks/volatility of the underlying market. So, to minimize the impacts of market risk, we focus on picking stocks with low beta (i.e. when the market moves in a direction, these stocks move less in that direction, compared to others). This is important for our client since they are buying and holding. During a market downturn which they cannot exit their positions, we want to minimize the potential losses.
- Crash count: This is a future-looking statistic we created. While std already looks at volatility, we specifically want to capture company risks that sink value quickly. If a stock loses 20% in a week due to company-specific new or shocks, such as earnings reports, similar events could happen again in the future. We therefore penalize these stocks to avoiding investing in them when possible.
- Market cap: Large and mature companies often have steady revenue streams, earnings, and are usually stabler than penny stocks. Specifically, we don't want to invest in a penny stock, startup, or similar company, that is simply stagnant or has low volume. On paper, these stocks may have low volatility, but their price could change dramatically in the future. So, instead we focus on established businesses that are less likely to face such events. 

#### Stage 3 - Risk score ranking

**What it does:** Each stock is given a risk score, where a lower score represents lower estimated risk.

> Risk score = 0.3 * (std z score) + 0.7 * (beta z-score) + 10 × (number of crashes) + 10 × (1 if market cap < $20B CAD)

Stocks are sorted from safest to riskiest.

**Why these choices:**

- Z-scores instead of raw values: since std and beta are measured differently and have different ranges (std is 0 to infinity, beta is -infinity to +infinity), we first standardize by converting std/beta values to a z-score (a measure of how far a value is from the sample mean). This standardization also lets us use z-score to compare whether an individual stock is more/less volatile than the average std at z-score = 0. It is calculated as follows:
> ![{776FE838-10BC-4151-AB97-AB5DE30342B0}.png](attachment:34484b08-885a-413a-a315-11a07fbe6ed4.png)
- Beta weighted more than std: at this stage, we are focusing on future-proofing our strategy against market downturns. We will focus more on optimizing std in stages 4 and 5 by using these rankings.
- Disproportionate penalties on stocks: We chose large penalties so that stocks with past crashes count or those small market caps are more likely to rank below those safer stocks. We penalize these stocks but not removing them directly since we need more stocks to meet the min portfolio requirments.
#### Stage 4 - Stock selection algorithm

**What it does:** Starting from the lowest-risk stock, we repeatedly add the next stock that lowers std of the new combined portfolio the most. We start by only considering next stocks with a risk score below -0.5 (much safer than average). Only up to 4 stocks per sector may be chosen. If under these rules, fewer than 10 stocks are chosen, the risk score cutoff is loosened in steps of 0.1, to ensure penalized stocks are only a last resort, and safer stocks are considered first.

**Why these choices:**
- Choosing by portfolio std: std of a portfolio is not equal to the average of the stocks individual std. It is actually less than this, which is the benefit of diversification, and this idea leads the whole picking process. The volatility of the portfolio also depends on how stocks move together (eg. stock A zigs while stock B zags, because they are in contending industries). This effect can additionally lower portfolio std, and the picker captures these offsetting relationships that raw stat observations don't.
- Equal weighting combined portfolios: when judging stock picks, we compare the std of the combined equal-weighted portfolio chosen so far plus that candidate. As more stocks are added, the weight to each one in consideration decreases. At first, the picker favours stabler stocks, while later, it focuses more on how the stock behaves alongside the stocks already selected.
- Starting from lowest risk scored stock: as seen in class, adding a riskier stock to a stabler stock in a portfolio can raise the std of the overall portfolio, from the std of the stabler stock alone. To account for this, we start off with the lowest risk stock and add stocks one at a time, choosing the stock which gives us the lowest combined portfolio std at each round. Sometimes this increases or decreases the combined portfolio's std, which is compared to the previous safest alternative portfolio.
- Score cutoff of -0.5: Testing with the picker, we found that without a cutoff, it picks the riskiest stocks because they often offset the calmer ones. We want to avoid choosing these risky stocks too early, so we restrict the picker to safer ones first, with safer ones still possibly offsetting each other. Gradually loosening it ensures at least 10 stocks are chosen.
- 4 stocks per sector: In the most restrictive case, we want at most 40% of 10 stocks in any sector. So this cap helps meet portfolio requirements (40% of 10 = 4). It also forces the portfolio to hold across industries and protect against sector-specific events/shocks (eg., regulation changes, outlook of an industry, etc.).  

#### Stage 5 - Weight optimization

**What it does:** Starting from equal weights, the optimizer tries moving 1% weight from one stock to another. It accepts the change if it lowers overall portfolio volatility over a historical 1-year holding period. Thi process continues until no further 1% weight transfer reduces the volatility. Then, we shift the period forward by 1 month and repeat the process. Throughout the process, each stock keeps at least 2% weight, no stock can exceed 15%, and no sector can exceed 40%. We also reject any weight change that brings the estimated annual return below the inflation rate. Finally we average the weights from all 1-year periods to have our final weights.

**Why these choices:**
- Buy and hold: We simulate buying and holding portfolios in the past. since stoke prices change over time, their weights in the portfolio also change. Our strategy optimizes weights to work well across many different long-term periods. We want to account for this when choosing our initial weights. If we only optimized for a single 1 year period, then weights might depend to heavily on the events within that period. Considering multiple periods ensures us to find weights that work well under different market conditions.
- 2% minimum weight: It prevents stocks from receiving weights close to 0% and avoids relying too heavily on a few core stocks and helps to keep diversification.
- Comparison to inflation rate: Client B wants to conserve as much money as possible, over a long period of time. If they just held completely unvolatile low growth stocks, they will still have a negative real rate of return, i.e. they lose money due to inflation over time. So we consider inflation when adjusting our portfolio weights. In tests the portfolio already had relatively high nominal returns so this restriction may not have had much impact. However, if a trul low volatility portfolio is achieved with new tickers, real returns should be kept positive.

**Comparison to equal weighted portfolio:**
- We compare our optimized portfolio with the equal weighted portfolio during same historical periods and sample stocks. This comparison shows how our weight adjustments actually improve the portfolio's performance.

**What evidence supports your strategy?** 
- Our optimized portfolio reduced daily std from 0.7359% to 0.6525%, which brings a decrease of about 11.33%. Also, the historical return increased from 59.63% to 72.77%. Both demostrates that over the period, our weight adjustments reduced volatility without losing returns. Our portfolio is effective for client B.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

- The past may not represent the future: all of our stats come from 1 period of past data, for simplification. For instance, the data does not include a full market crisis like 2008, which could be good for informing how this portfolio would perform in such a sever downturn, as our client wants a long-term portfolio and strategy. Our beta-based ranking can reduce this risk but cannot completely remove market risk.

- Concentration in sectors: Our portfolio in testing ended up with many stocks in consumer staples, healthcare, large banks, and telecom. These have historically been stable but share risks. 

- Many of our holdings are US stocks, so their CAD value depends on the USD/CAD exchange rate. An increase in the Canadian dollar would lower the CAD value of these positions even if their US prices didn't change.

- Judgment: many parameters like the 0.3/0.7 weighting are judgment calls and not proven optimal values.

- Crash potential: a stock that has never crashed in the past still has the potential to do so in the future.

- Large cap volume: large-caps have higher volume and inherently more volatility on any trading day than a small-cap company does. While this increaes long-term / over-time stability, it may result in high short-term volatility, as shares trade hands more often.

- Equal weight selection: Our picker algorithm uses equal weights to decide stocks to combine into portfolio next. This may overlook stocks that can outperform others at a non-equal weight, or can deliver a better opposite directional benefit when weighted lower amount. But realistically, it would be really hard to considering different weights during stock selection here, given portfolio restrctions and an unkown number of stocks we will have in the final portfolio.